# Import Model 

In [10]:
import os
import re
import copy
import random
import numpy as np
import torch
import matplotlib.pyplot as plt
import pandas as pd
from collections import Counter

# necessary for extending the vocabulary
import pickle
# necessary for importing the model
import sys
sys.path.append("nanoGPT")
from model import GPTConfig, GPT

# Load model S from task 1
BASE = "data/shakespeare_char"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

block_size = 256
batch_size = 64
learning_rate = 5e-5
max_iters = 2000
eval_iters = 100

# load checkpoint of the model
ckpt = torch.load("out-shakespeare-S-split100\\ckpt.pt", map_location=DEVICE)
model = GPT(GPTConfig(**ckpt["model_args"]))
state_dict = ckpt["model"]
# remove _origin_mod because our model has been saved with torch.compile()
state_dict = {k.replace("_orig_mod.", ""): v for k, v in state_dict.items()}
model.load_state_dict(state_dict)

model.to(DEVICE)
model.eval()

number of parameters: 0.80M


GPT(
  (transformer): ModuleDict(
    (wte): Embedding(65, 128)
    (wpe): Embedding(256, 128)
    (drop): Dropout(p=0.2, inplace=False)
    (h): ModuleList(
      (0-3): 4 x Block(
        (ln_1): LayerNorm()
        (attn): CausalSelfAttention(
          (c_attn): Linear(in_features=128, out_features=384, bias=False)
          (c_proj): Linear(in_features=128, out_features=128, bias=False)
          (attn_dropout): Dropout(p=0.2, inplace=False)
          (resid_dropout): Dropout(p=0.2, inplace=False)
        )
        (ln_2): LayerNorm()
        (mlp): MLP(
          (c_fc): Linear(in_features=128, out_features=512, bias=False)
          (gelu): GELU(approximate='none')
          (c_proj): Linear(in_features=512, out_features=128, bias=False)
          (dropout): Dropout(p=0.2, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm()
  )
  (lm_head): Linear(in_features=128, out_features=65, bias=False)
)

# Get the validation loss pre training
because afterwards most parameters of the model are overwritten and we get an invalid value

In [11]:
# Define the batch sampler
def get_batch(data):
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([torch.tensor(data[i:i+block_size]) for i in ix])
    y = torch.stack([torch.tensor(data[i+1:i+block_size+1]) for i in ix])
    return x.to(DEVICE), y.to(DEVICE)

# Compute validation loss, appended .astype(...) because we need long
val_data = np.memmap(
    f"{BASE}/val.bin",
    dtype=np.uint16,
    mode="r"
).astype(np.int64)

def estimate_val_loss(model):
    model.eval()
    losses = []

    for _ in range(eval_iters):
        xb, yb = get_batch(val_data)
        with torch.no_grad():
            _, loss = model(xb, yb)
        losses.append(loss.item())

    return sum(losses) / len(losses)

model = model.to(DEVICE)
val_pre  = estimate_val_loss(model)
val_pre

1.582333995103836

# Extend vocabulary and embedding to fit our new special tokens "[" and "]"
This ensures that the model knows all tokens relevant for the tasks.

In [12]:
# extend vocabulary to fit our specially desigend tokens
meta_path = os.path.join(BASE, "meta.pkl")
with open(meta_path, "rb") as f:
    meta = pickle.load(f)

stoi = meta["stoi"]
itos = meta["itos"]

# Add special tokens
special_text = "[SPEAKER][ANSWER][END][CLASSIFY]"
for ch in special_text:
    if ch not in stoi:
        stoi[ch] = len(stoi)
        itos[len(itos)] = ch

vocab_size = len(stoi)

# extend embedding to fit the new tokens needed for our task "[", "]"
old_vocab_size = model.config.vocab_size
new_vocab_size = len(stoi)

if new_vocab_size > old_vocab_size:
    old_emb = model.transformer.wte.weight.data
    new_emb = torch.nn.Embedding(new_vocab_size, model.config.n_embd)
    new_emb.weight.data[:old_vocab_size] = old_emb
    model.transformer.wte = new_emb

    old_lm = model.lm_head.weight.data
    new_lm = torch.nn.Linear(model.config.n_embd, new_vocab_size, bias=False)
    new_lm.weight.data[:old_vocab_size] = old_lm
    model.lm_head = new_lm

    model.config.vocab_size = new_vocab_size

# Define functions for encoding and decoding
def encode(text):
    return [stoi[c] for c in text]

def decode(indices):
    return "".join([itos[i] for i in indices])


# Build dataset for speaker classification
Transform the shakespeare input to create a train and test set for task A. 

In [13]:

with open(f"{BASE}/input.txt", "r") as f:
    lines = f.readlines()

# regex to identify task specific words
speaker_pattern = re.compile(r"^([A-Z][A-Z ]+)[\.:]")

data = []
current_speaker = None

for line in lines:
    line = line.strip()
    match = speaker_pattern.match(line)
    if match:
        current_speaker = match.group(1)
    elif current_speaker and line:
        data.append((line, current_speaker))

speaker_counts = Counter([s for _, s in data])
top_10 = [s for s, _ in speaker_counts.most_common(10)]

filtered = [(l, s) for l, s in data if s in top_10]

taskA_examples = [
    f"[SPEAKER] {l} [ANSWER] {s} [END]"
    for l, s in filtered
]

# Split into train/test
random.shuffle(taskA_examples)

taskA_train = taskA_examples[:500]
taskA_test = taskA_examples[500:600]

# Save as text files
with open(f"{BASE}/sft_speaker_train.txt", "w") as f:
    for ex in taskA_train:
        f.write(ex + "\n")

with open(f"{BASE}/sft_speaker_test.txt", "w") as f:
    for ex in taskA_test:
        f.write(ex + "\n")

# Build dataset for verse/prose classification

In [14]:

# helper function checks if a block of text is likely to be a verse based on line lengths and number of lines
def is_verse(block):
    lines = block.split("\n")
    avg_len = sum(len(l) for l in lines) / max(len(lines), 1)
    return avg_len < 60 and len(lines) >= 3
    
blocks = open(f"{BASE}/input.txt").read().split("\n\n")

taskB_examples = []

for b in blocks:
    if len(b.strip()) < 50:
        continue
    label = "VERSE" if is_verse(b) else "PROSE"
    text = f"[CLASSIFY] {b} [ANSWER] {label} [END]"
    taskB_examples.append(text)

# Split into train/test
random.shuffle(taskB_examples)
taskB_train = taskB_examples[:500]
taskB_test  = taskB_examples[500:600]

# Save as text files
with open(f"{BASE}/sft_classify_train.txt", "w") as f:
    for ex in taskB_train:
        f.write(ex + "\n")

with open(f"{BASE}/sft_classify_test.txt", "w") as f:
    for ex in taskB_test:
        f.write(ex + "\n")


# Define the training function

In [15]:

# Convert datasets into training format
def build_array(text_list):
    text = "\n".join(text_list)
    return np.array(encode(text), dtype=np.int64)

# Training function
def fine_tune(base_model, train_data, title, lr=learning_rate, max_iters=max_iters, save_plot=False):
    model = copy.deepcopy(base_model)
    model.to(DEVICE)
    model.train()

    optimizer = torch.optim.AdamW(model.parameters(), lr=lr)

    losses = []

    for step in range(max_iters):
        xb, yb = get_batch(train_data)

        logits, loss = model(xb, yb)
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

        losses.append(loss.item())

        if step % 200 == 0:
            print(f"{title} | step {step} | loss {loss.item():.4f}")

    if save_plot:
        plt.plot(losses)
        plt.title(title)
        plt.xlabel("Step")
        plt.ylabel("Loss")
        plt.savefig(f"train_plots/{title}_loss.png")
        plt.close()

    return model, losses

# Define evaluation functions
Here, we let the model do task A and task B. [ANSWER], [SPEAKER] and [END] blocks are then removed to compare the prediction and the actual label.

In [16]:

# assumption: the model correctly generates the answer in the format "[ANSWER] ... [END]"
def extract_answer(text):
    if "[ANSWER]" in text:
        return text.split("[ANSWER]")[1].split("[END]")[0].strip()
    return ""

def compute_taskA_accuracy(model):

    model.eval()
    correct = 0
    longest_label = 0
    # find longest label for max_new_tokens
    for ex in taskA_test:
        line = ex.split("[SPEAKER]")[1].split("[ANSWER]")[0].strip()
        label = extract_answer(ex)
        if len(label) > longest_label:
            longest_label = len(label)

    for ex in taskA_test:
        line = ex.split("[SPEAKER]")[1].split("[ANSWER]")[0].strip()
        label = extract_answer(ex)

        prompt = f"[SPEAKER] {line} [ANSWER] "
        idx = torch.tensor(encode(prompt)).unsqueeze(0).to(DEVICE)

        out = model.generate(idx, max_new_tokens=longest_label + len(" [END]"))
        pred_text = decode(out[0].tolist())
        pred = extract_answer(pred_text)
        print(f"pred={pred:20s} | label={label:20s} | pred_text={pred_text}")
        if pred == label:
            correct += 1

    return correct / len(taskA_test)

def compute_taskB_accuracy(model):

    model.eval()
    correct = 0

    for ex in taskB_test:
        label = extract_answer(ex)
        prompt = ex.split("[ANSWER]")[0] + "[ANSWER] "
        idx = torch.tensor(encode(prompt)).unsqueeze(0).to(DEVICE)
        # VERSE and PROSE have the same length
        out = model.generate(idx, max_new_tokens=len("VERSE [END]"))
        pred_text = decode(out[0].tolist())
        pred = extract_answer(pred_text)

        print(f"pred={pred:20s} | label={label:20s} | pred_text={pred_text}")
        if pred == label:
            correct += 1

    return correct / len(taskB_test)

Create the data and finetune a model for task A, B and both

In [17]:
A_train_data = build_array(taskA_train)
A_test_data  = build_array(taskA_test)

B_train_data = build_array(taskB_train)
B_test_data  = build_array(taskB_test)

# Run experiments
Finally, using the previously defined functions, run the model on task A, task B and the multitask and measure performance. Iterate over hyperparameters to find the best combination.

In [18]:

import json

results_list = []

# Track best models and accuracies
best_acc_A_single = 0
best_acc_B_single = 0
best_acc_A_multi = 0
best_acc_B_multi = 0

best_model_A_single = None
best_model_B_single = None
best_model_A_multi = None
best_model_B_multi = None

for lr in [1e-4, 5e-5]:
    for max_iter in [1000, 2000, 3000]:
        print(f"Learning Rate: {lr}")
        print(f"Max Iterations: {max_iter}")
        print("Finetuning Single Task A")
        model_A, _ = fine_tune(model, A_train_data, f"Single_Task_A{lr}_{max_iter}", lr=lr, max_iters=max_iter, save_plot=True)

        print("Finetuning Single Task B")
        model_B, _ = fine_tune(model, B_train_data, f"Single_Task_B{lr}_{max_iter}", lr=lr, max_iters=max_iter, save_plot=True)

        print("Finetuning Multi Task A+B")
        # randomize training data for both 
        combined = taskA_train + taskB_train
        random.shuffle(combined)
        mixed = build_array(combined)
        model_multi, _ = fine_tune(model, mixed, f"Multi_Task{lr}_{max_iter}", lr=lr, max_iters=max_iter, save_plot=True)
        
        print("Evaluating (averaging over 5 runs)")
        
        # Compute accuracies 5 times and average
        acc_A_single_list = [compute_taskA_accuracy(model_A) for _ in range(5)]
        acc_A_single = sum(acc_A_single_list) / len(acc_A_single_list)
        
        acc_B_single_list = [compute_taskB_accuracy(model_B) for _ in range(5)]
        acc_B_single = sum(acc_B_single_list) / len(acc_B_single_list)
        
        acc_A_multi_list = [compute_taskA_accuracy(model_multi) for _ in range(5)]
        acc_A_multi = sum(acc_A_multi_list) / len(acc_A_multi_list)
        
        acc_B_multi_list = [compute_taskB_accuracy(model_multi) for _ in range(5)]
        acc_B_multi = sum(acc_B_multi_list) / len(acc_B_multi_list)

        val_A    = estimate_val_loss(model_A)
        val_B    = estimate_val_loss(model_B)
        val_multi = estimate_val_loss(model_multi)
        
        # Check and save best models
        if acc_A_single > best_acc_A_single:
            best_acc_A_single = acc_A_single
            best_model_A_single = copy.deepcopy(model_A)
            print(f"New best A single: {acc_A_single:.4f}")
        
        if acc_B_single > best_acc_B_single:
            best_acc_B_single = acc_B_single
            best_model_B_single = copy.deepcopy(model_B)
            print(f"New best B single: {acc_B_single:.4f}")
        
        if acc_A_multi > best_acc_A_multi:
            best_acc_A_multi = acc_A_multi
            best_model_A_multi = copy.deepcopy(model_multi)
            print(f"New best A multi: {acc_A_multi:.4f}")
        
        if acc_B_multi > best_acc_B_multi:
            best_acc_B_multi = acc_B_multi
            best_model_B_multi = copy.deepcopy(model_multi)
            print(f"New best B multi: {acc_B_multi:.4f}")
        
        # Store results
        results_list.append({
            "learning_rate": lr,
            "max_iters": max_iter,
            "acc_A_single": acc_A_single,
            "acc_B_single": acc_B_single,
            "acc_A_multi": acc_A_multi,
            "acc_B_multi": acc_B_multi,
            "val_A": val_A,
            "val_B": val_B,
            "val_multi": val_multi
        })

# Save to JSON file
with open("hyperparameter_results.json", "w") as f:
    json.dump(results_list, f, indent=2)

print(f"\nSaved {len(results_list)} results to hyperparameter_results.json")
print(f"Best accuracies: A_single={best_acc_A_single:.4f}, B_single={best_acc_B_single:.4f}, A_multi={best_acc_A_multi:.4f}, B_multi={best_acc_B_multi:.4f}")


Learning Rate: 0.0001
Max Iterations: 1000
Finetuning Single Task A
Single_Task_A0.0001_1000 | step 0 | loss 3.7966
Single_Task_A0.0001_1000 | step 200 | loss 0.9132
Single_Task_A0.0001_1000 | step 400 | loss 0.8443
Single_Task_A0.0001_1000 | step 600 | loss 0.7725
Single_Task_A0.0001_1000 | step 800 | loss 0.7603
Finetuning Single Task B
Single_Task_B0.0001_1000 | step 0 | loss 2.4209
Single_Task_B0.0001_1000 | step 200 | loss 1.3648
Single_Task_B0.0001_1000 | step 400 | loss 1.3143
Single_Task_B0.0001_1000 | step 600 | loss 1.2958
Single_Task_B0.0001_1000 | step 800 | loss 1.2500
Finetuning Multi Task A+B
Multi_Task0.0001_1000 | step 0 | loss 2.6365
Multi_Task0.0001_1000 | step 200 | loss 1.2991
Multi_Task0.0001_1000 | step 400 | loss 1.2333
Multi_Task0.0001_1000 | step 600 | loss 1.1742
Multi_Task0.0001_1000 | step 800 | loss 1.2222
Evaluating (averaging over 5 runs)
pred=GLOUCESTER           | label=KING RICHARD II      | pred_text=[SPEAKER] Cry woe, destruction, ruin and decay: [A

In [ ]:
# Load and display results from hyperparameter tuning
import json
import pandas as pd

with open("hyperparameter_results.json", "r") as f:
    results_list = json.load(f)

# Create DataFrame from saved results
results_df = pd.DataFrame(results_list)

# Display full results table
print("Hyperparameter Tuning Results:")
print(results_df.to_string(index=False))

# Optional: Create a pivot table showing accuracy for each hyperparameter combo
print("\n\nTask A Accuracy (Single Model):")
pivot_A_single = results_df.pivot_table(
    values='acc_A_single', 
    index='max_iters', 
    columns='learning_rate'
)
print(pivot_A_single)

print("\n\nTask B Accuracy (Single Model):")
pivot_B_single = results_df.pivot_table(
    values='acc_B_single', 
    index='max_iters', 
    columns='learning_rate'
)
print(pivot_B_single)

print("\n\nMulti-Task - Task A Accuracy:")
pivot_A_multi = results_df.pivot_table(
    values='acc_A_multi', 
    index='max_iters', 
    columns='learning_rate'
)
print(pivot_A_multi)

print("\n\nMulti-Task - Task B Accuracy:")
pivot_B_multi = results_df.pivot_table(
    values='acc_B_multi', 
    index='max_iters', 
    columns='learning_rate'
)
print(pivot_B_multi)


Hyperparameter Tuning Results:
 learning_rate  max_iters  acc_A_single  acc_B_single  acc_A_multi  acc_B_multi    val_A    val_B  val_multi
       0.00010       1000         0.114         0.658        0.080        0.726 2.390233 1.630383   1.626479
       0.00010       2000         0.114         0.668        0.114        0.826 2.880009 1.656841   1.649319
       0.00010       3000         0.120         0.858        0.114        0.848 3.441619 1.690353   1.668628
       0.00005       1000         0.078         0.686        0.080        0.650 2.162603 1.599329   1.616468
       0.00005       2000         0.126         0.674        0.078        0.788 2.426398 1.617135   1.617574
       0.00005       3000         0.114         0.668        0.116        0.824 2.713657 1.634200   1.625852


Task A Accuracy (Single Model):
learning_rate  0.00005  0.00010
max_iters                      
1000             0.078    0.114
2000             0.126    0.114
3000             0.114    0.120


Task B Acc

: 

Using the best hyperparameters, this is the results table:

In [20]:
# Generate samples using the best models

def generate_unconditional(model):
    model = model.to(DEVICE)
    idx = torch.zeros((1,1), dtype=torch.long).to(DEVICE)
    out = model.generate(idx, max_new_tokens=300)
    return decode(out[0].tolist())

with open("generated_samples.txt", "w") as f:
    f.write("=== PRETRAINED ===\n")
    f.write(generate_unconditional(model))
    f.write("\n\n=== BEST SINGLE A ===\n")
    f.write(generate_unconditional(best_model_A_single))
    f.write("\n\n=== BEST SINGLE B ===\n")
    f.write(generate_unconditional(best_model_B_single))
    f.write("\n\n=== BEST MULTI A ===\n")
    f.write(generate_unconditional(best_model_A_multi))
    f.write("\n\n=== BEST MULTI B ===\n")
    f.write(generate_unconditional(best_model_B_multi))
